In [ ]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from tqdm import tqdm
import numpy as np

from monai.networks.nets import SwinUNETR
from monai.transforms import (
    Compose, LoadImaged, SpatialPadd, RandSpatialCropd, RandZoomd, ToTensord
)
from monai.data import DataLoader, Dataset

# ==========================================
# 1. 优化版 Independent Masking MAE Wrapper
# ==========================================
class MultiModal_SwinUNETR_MAE(nn.Module):
    def __init__(self, img_size=(96, 128, 128), patch_size=(12, 16, 16), in_channels=2, mask_ratio=0.5):
        super().__init__()
        self.mask_ratio = mask_ratio
        self.patch_size = patch_size
        
        self.encoder_decoder = SwinUNETR(
            img_size=img_size,                
            in_channels=in_channels,          
            out_channels=in_channels,         
            feature_size=96,                  
            depths=(2, 2, 18, 2),              
            num_heads=(3, 6, 12, 24),         
            norm_name='instance',             
            drop_rate=0.0,                    
            attn_drop_rate=0.0,               
            dropout_path_rate=0.05,           
            normalize=True,
            use_checkpoint=True,              
            spatial_dims=3,
            downsample='mergingv2',           
            use_v2=True                       
        )
        
        # [优化 1]：移除可学习的 Mask Token。因为数据已做 Z-Score，
        # 直接用 0.0 (即均值) 填充 Mask 区域，这不仅消除了块状伪影边缘，还符合最优数学期望。

    def forward(self, x):
        B, C, Z, Y, X = x.shape
        pz, py, px = self.patch_size
        
        grid_z, grid_y, grid_x = Z // pz, Y // py, X // px
        
        # 为 PET 和 CT 独立生成 Mask (B, C, 8, 8, 8)
        noise = torch.rand(B, C, grid_z, grid_y, grid_x, device=x.device)
        mask = (noise < self.mask_ratio).float() 
        
        # 放大 Mask 回体素级分辨率 (B, 2, 96, 128, 128)
        mask_expanded = mask.repeat_interleave(pz, dim=2).repeat_interleave(py, dim=3).repeat_interleave(px, dim=4)
        
        # [优化 1 继续]：均值插补 (Mean Imputation)。直接把被 Mask 的地方变为 0.0
        x_masked = x * (1 - mask_expanded)
        
        # 送入网络重构
        x_rec = self.encoder_decoder(x_masked)
        
        # [优化 2]：加权全局损失函数 (Weighted Global Loss)
        # 计算每个像素的 MSE
        mse_loss = F.mse_loss(x_rec, x, reduction='none')
        
        # Mask 区域的 Loss (核心学习目标，权重 1.0)
        loss_masked = (mse_loss * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
        
        # 非 Mask 区域的 Loss (用于稳定 U-Net 的跳跃连接，防止图像崩坏，权重 0.2)
        loss_unmasked = (mse_loss * (1 - mask_expanded)).sum() / ((1 - mask_expanded).sum() + 1e-8)
        
        # 总 Loss
        loss = loss_masked + 0.2 * loss_unmasked
        
        return loss, x_rec

# ==========================================
# 2. Dataset & Transforms (保持您的设定)
# ==========================================
def get_pretraining_dataloader(batch_size=16):
    base_path = Path('/gluon4/xl693/PETCTfoundation/')
    anomalies = [
        'LDca4f40/LDca5687', 'LDca56d5/LDca5e13', 'LDca4eed/LDca54ed',
        'LDca519f/LDca5602', 'LDca56da/LDca5d8b', 'LDca58c7/LDca5c77',
        'LDca58c3/LDca5c6f', 'LDca4f3f/LDca5507', 'LDca58c2/LDca5c6e',
        'LDca5163/LDca5581', 'LDca56d2/LDca5d2b'
    ]
    
    data_dicts = []
    for dataset_dir in sorted(base_path.iterdir()):
        if not dataset_dir.is_dir(): continue
        for f in dataset_dir.rglob('*.npy'):
            if any(anomaly in f.as_posix() for anomaly in anomalies):
                continue
            data_dicts.append({"image": f.as_posix()})
            
    print(f"Total valid 3D volumes loaded: {len(data_dicts)}")

    train_transforms = Compose([
        LoadImaged(keys=["image"], reader="NumpyReader"), 
        RandZoomd(keys=["image"], prob=0.5, min_zoom=0.9, max_zoom=1.1, mode="trilinear", keep_size=False),
        SpatialPadd(keys=["image"], spatial_size=(96, 128, 128), mode="constant", constant_values=0),
        RandSpatialCropd(keys=["image"], roi_size=(96, 128, 128), random_size=False),
        ToTensord(keys=["image"])
    ])

    dataset = Dataset(data=data_dicts, transform=train_transforms)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    return dataloader

# ==========================================
# 3. 健壮的训练循环 (含 LR Scheduler & Grad Clip)
# ==========================================
def train_foundation_model():
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using GPUs: {os.environ['CUDA_VISIBLE_DEVICES']}")
    
    model = MultiModal_SwinUNETR_MAE(
        img_size=(96, 128, 128), 
        patch_size=(12, 16, 16), 
        in_channels=2, 
        mask_ratio=0.5
    )
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    model = model.to(device)
    
    # 基础学习率设定，配合后续的 Scheduler
    optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
    
    num_epochs = 200
    
    # [优化 3]：引入 Cosine 退火学习率for Transformer
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)
    
    scaler = GradScaler('cuda')
    dataloader = get_pretraining_dataloader(batch_size=16)
    
    start_epoch = 0
    best_loss = float('inf')
    latest_ckpt_path = "swin_mae_latest_v2.pth"
    best_ckpt_path = "swin_mae_best_v2.pth"
    
    if os.path.exists(latest_ckpt_path):
        print(f"=> Loading checkpoint from '{latest_ckpt_path}'")
        checkpoint = torch.load(latest_ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scaler.load_state_dict(checkpoint['scaler_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict']) # 恢复学习率状态
        start_epoch = checkpoint['epoch']
        best_loss = checkpoint['best_loss']
        print(f"=> Resumed from epoch {start_epoch+1}")
    else:
        print("=> Starting training from scratch")

    for epoch in range(start_epoch, num_epochs):
        model.train()
        epoch_loss = 0.0
        
        # 获取当前 Epoch 的学习率
        current_lr = optimizer.param_groups[0]['lr']
        progress_bar = tqdm(enumerate(dataloader), total=len(dataloader), desc=f"Epoch {epoch+1}/{num_epochs} [LR: {current_lr:.2e}]")
        
        for step, batch_data in progress_bar:
            inputs = batch_data["image"].to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            
            with autocast('cuda'):
                loss, _ = model(inputs)
                loss = loss.mean()
            
            scaler.scale(loss).backward()
            
            # [优化 4]：梯度裁剪。防止跨模态 Mask 早期产生的剧烈梯度炸毁网络权重
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            scaler.step(optimizer)
            scaler.update()
            
            epoch_loss += loss.item()
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}"})
            
        # 更新学习率
        scheduler.step()
            
        avg_loss = epoch_loss / len(dataloader)
        print(f"\n[Epoch {epoch+1}] Average Global Loss: {avg_loss:.4f}\n")
        
        state_dict = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_loss': best_loss
        }
        
        torch.save(state_dict, latest_ckpt_path)
        
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save(state_dict, best_ckpt_path)
            print(f"*** New Best Model Saved! Loss: {best_loss:.4f} ***")

if __name__ == '__main__':
    train_foundation_model()